In [1]:
# Neural Identifier Training - Differential Drive Robot
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (Differential Drive Robot)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a differential drive robot.
    state = [x, y, θ, v, ω] 
        x, y: position (m)
        θ: orientation (rad)
        v: linear velocity (m/s)
        ω: angular velocity (rad/s)
    u = [v_cmd, ω_cmd]: velocity commands
    """
    # Robot Parameters
    m = 2.0          # Mass (kg)
    I = 0.5          # Moment of inertia (kg⋅m²)
    b_v = 0.5        # Linear drag coefficient
    b_ω = 0.3        # Angular drag coefficient
    tau_v = 0.2      # Time constant for linear velocity
    tau_ω = 0.15     # Time constant for angular velocity
    
    x, y, θ, v, ω = state
    v_cmd, ω_cmd = u
    
    # Kinematic equations (position and orientation)
    x_dot = v * np.cos(θ)
    y_dot = v * np.sin(θ)
    θ_dot = ω
    
    # Dynamic equations (velocities with first-order dynamics)
    # Modelo simplificado: τ*dv/dt + v = v_cmd
    v_dot = (v_cmd - v) / tau_v - (b_v / m) * v * np.abs(v)  # Con fricción no lineal
    ω_dot = (ω_cmd - ω) / tau_ω - (b_ω / I) * ω * np.abs(ω)  # Con fricción no lineal
    
    return np.array([x_dot, y_dot, θ_dot, v_dot, ω_dot])

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-3):
    """
    RK4 integration step for better accuracy.
    """
    # RK4 para mayor precisión
    k1 = plant_dynamics(x_k, u_k)
    k2 = plant_dynamics(x_k + 0.5*dt*k1, u_k)
    k3 = plant_dynamics(x_k + 0.5*dt*k2, u_k)
    k4 = plant_dynamics(x_k + dt*k3, u_k)
    
    x_kp1 = x_k + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    
    # Add small process noise (Laplacian for heavier tails)
    noise = np.random.normal(0, process_noise_std, size=5)
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=0.5):
    """Sigmoid S(z)."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for Differential Drive Robot - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [x, y, θ, v, ω]
    u_input = [v_cmd, ω_cmd]
    neuron_index: índice de la neurona (0=x, 1=y, 2=θ, 3=v, 4=ω)
    """
    x, y, θ, v, ω = x_est
    v_cmd, ω_cmd = u_input
    
    # Términos básicos sigmoidales
    s_x = sigmoidal(x)
    s_y = sigmoidal(y)
    s_θ = sigmoidal(θ)
    s_v = sigmoidal(v)
    s_ω = sigmoidal(ω)
    
    # Términos trigonométricos (importantes para la cinemática)
    cos_θ = np.cos(θ)
    sin_θ = np.sin(θ)
    
    # Comandos escalados
    s_v_cmd = sigmoidal(v_cmd)
    s_ω_cmd = sigmoidal(ω_cmd)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para x (posición horizontal)
        # dx/dt = v*cos(θ)
        return np.array([
            s_x,                       # Velocidad lineal
            # cos_θ,                     # Componente direccional
            # s_v * cos_θ,              # Término cinemático principal
            s_θ,                       # Orientación
            s_v * s_θ,                # Interacción velocidad-orientación
            s_v**2,                    # Término cuadrático
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 1:  # Neurona para y (posición vertical)
        # dy/dt = v*sin(θ)
        return np.array([
            # s_v,                       # Velocidad lineal
            # sin_θ,                     # Componente direccional
            # s_v * sin_θ,              # Término cinemático principal
            s_θ,                       # Orientación
            s_v * s_θ,                # Interacción velocidad-orientación
            s_y**2,                    # Término cuadrático
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 2:  # Neurona para θ (orientación)
        # dθ/dt = ω
        return np.array([
            s_ω,                       # Velocidad angular (término principal)
            s_ω**2,                    # Término cuadrático
            # s_ω**3,                    # Término cúbico (no linealidad)
            s_v * s_ω,                # Acoplamiento con velocidad lineal
            s_θ,                       # Orientación actual
            # s_ω_cmd * 0.2,            # Comando de velocidad angular
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 3:  # Neurona para v (velocidad lineal)
        # dv/dt = (v_cmd - v)/tau - friction
        return np.array([
            s_v,                       # Estado actual
            s_v**2,                    # Fricción cuadrática
            s_v**3,                    # Fricción cúbica
            # v_cmd * 0.1,              # Comando (lineal, no saturado)
            s_v_cmd,                   # Comando (sigmoidal)
            # s_v * s_v_cmd,            # Interacción estado-comando
            s_ω,                       # Acoplamiento con velocidad angular
            # s_v * np.abs(v),          # Término de fricción absoluta
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 4:  # Neurona para ω (velocidad angular)
        # dω/dt = (ω_cmd - ω)/tau - friction
        return np.array([
            s_ω,                       # Estado actual
            # s_ω**2,                    # Fricción cuadrática
            s_ω**3,                    # Fricción cúbica
            # ω_cmd * 0.1,              # Comando (lineal, no saturado)
            # s_ω_cmd,                   # Comando (sigmoidal)
            s_ω * s_ω_cmd,            # Interacción estado-comando
            s_v,                       # Acoplamiento con velocidad lineal
            # s_ω * np.abs(ω),          # Término de fricción absoluta
            # 1.0                        # Bias
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:  # x
        return 4
    elif neuron_index == 1:  # y
        return 3
    elif neuron_index == 2:  # θ
        return 4
    elif neuron_index == 3:  # v
        return 5
    elif neuron_index == 4:  # ω
        return 4
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-3, R=1e-5):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            # Construir vector de características H_i (ecuación 8)
            z = construct_z_vector(x_k, u_k, i)
            H_i = z.reshape(-1, 1)
            
            # Error de identificación e_i(k) (ecuación 7)
            # e_i(k) = x_i(k) - X̂_i(k)
            x_hat_i = np.dot(self.weights[i], z)
            e_i = x_kp1[i] - x_hat_i
            
            # Ganancia de Kalman K_i(k) (ecuación 6)
            # K_i(k) = P_i(k) H_i(k) [R_i(k) + H_i(k) P_i(k) H_i(k)]^{-1}
            S = self.R + (H_i.T @ self.P[i] @ H_i)[0, 0]
            K_i = (self.P[i] @ H_i).flatten() / S
            
            # Actualización de pesos ω_i(k+1) (ecuación superior)
            # ω_i(k+1) = ω_i(k) + η_i K_i(k) e_i(k)
            self.weights[i] = self.weights[i] + self.eta * K_i * e_i
            
            # Actualización de covarianza P_i(k+1) (ecuación 6, tercera línea)
            # P_i(k+1) = P_i(k) - K_i(k) H_i(k) P_i(k) + Q_i(k)
            self.P[i] = self.P[i] - np.outer(K_i, H_i.flatten()) @ self.P[i] + self.Q_matrices[i]

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.3 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*1e-3 for i in range(n_neurons)]
        self.R = 1e-5
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_particles=1000):  # ✅ Más partículas
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        self.particles = [np.random.randn(n_particles, get_z_size(i))*0.3 
                         for i in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        self.R_std = 0.15          # ✅ Ruido de medición reducido
        self.Q_std = 0.055         
        self.regularization_std = 1.0e-5  # ✅ Jitter post-resampling

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n_weights = get_z_size(i)
            
            # 1. Drift con Q_std (no R_std!)
            self.particles[i] += np.random.randn(self.n_particles, n_weights) * self.Q_std
            
            # 2. Weight
            preds = self.particles[i] @ z
            err = x_kp1[i] - preds
            
            # Student-t distribution (heavier tails than Gaussian)
            # log p(y|x) ∝ -log(1 + (err/scale)²)
            nu = 3.0  # degrees of freedom (lower = heavier tails)
            scale = self.R_std * np.sqrt((nu - 2) / nu)  # scale parameter
            log_likelihood = -(nu + 1) / 2 * np.log(1 + (err / scale) ** 2)
            
            self.weights_pf[i] *= np.exp(log_likelihood - np.max(log_likelihood))
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            
            # 3. Resample con regularización
            eff_N = 1.0 / np.sum(self.weights_pf[i]**2)
            if eff_N < self.n_particles/5:  # ✅ Umbral más bajo
                indices = np.random.choice(self.n_particles, self.n_particles, 
                                         p=self.weights_pf[i])
                self.particles[i] = self.particles[i][indices]
                
                # ✅ REGULARIZACIÓN: Agregar jitter
                self.particles[i] += np.random.randn(self.n_particles, n_weights) * self.regularization_std
                
                self.weights_pf[i].fill(1.0/self.n_particles)
                
    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) 
                for i in range(self.n_neurons)]



In [2]:
print("="*80)
print(" "*20 + "SIMULATION 1: EKF-RHONN ONLY")
print("="*80)

                    SIMULATION 1: EKF-RHONN ONLY


In [3]:
import time

# ============================================================
# SIMULATION 1: EKF-RHONN ONLY
# ============================================================
if __name__ == "__main__":
    # Simulation parameters
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    n_states = 5  # [x, y, θ, v, ω]
    
    # Measurement noise parameters
    position_noise_std = 0.01
    angle_noise_std = 0.005
    velocity_noise_std = 0.02
    omega_noise_std = 0.003
    
    # Set independent random seed for EKF simulation
    np.random.seed(1001)  # Unique seed for EKF
    
    # Generate initial weights (uniform distribution)
    initial_weights_ekf = []
    for i in range(n_states):
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights_ekf.append(w_init.copy())
    
    print("\n[EKF] Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights_ekf):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}, mean={w.mean():.4f}")
    
    # Initialize EKF trainer
    ekf = EKF_Trainer(n_states, eta=1.0, P0=1.0, Q=1e-3, R=1e-2)
    
    # Set initial weights
    for i in range(n_states):
        ekf.weights[i] = initial_weights_ekf[i].copy()
    
    print("\n✓ EKF initialized with uniform RHONN weights")
    
    # Arrays for states
    x_true_ekf = np.zeros((n_steps, 5))
    x_est_ekf = np.zeros((n_steps, 5))
    y_measured_ekf = np.zeros((n_steps, 5))
    ekf_training_times = []
    
    # Initial conditions
    x_true_ekf[0] = [0.0, 0.0, 0.0, 0.0, 0.0]
    x_est_ekf[0] = x_true_ekf[0]
    
    # Add noise to initial measurement
    initial_noise = np.array([
        np.random.normal(0, position_noise_std),
        np.random.normal(0, position_noise_std),
        np.random.normal(0, angle_noise_std),
        np.random.normal(0, velocity_noise_std),
        np.random.normal(0, omega_noise_std)
    ])
    y_measured_ekf[0] = x_true_ekf[0] + initial_noise
    
    # Excitation input
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        v_cmd = 1.0 + 0.5 * np.sin(0.5 * t[k])
        ω_cmd = 0.8 * np.sin(1.0 * t[k])
        
        if 300 < k < 320:
            v_cmd = 2.0
            ω_cmd = 1.5
        elif 700 < k < 720:
            v_cmd = 0.5
            ω_cmd = -1.2
        elif 1100 < k < 1120:
            v_cmd = -0.8
            ω_cmd = 0.0
        
        u_hist[k] = [v_cmd, ω_cmd]
    
    print("\n[EKF] Starting simulation...")
    state_names = ['x (pos)', 'y (pos)', 'θ (orient)', 'v (lin vel)', 'ω (ang vel)']
    print("\nNeuron structure:")
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}): {get_z_size(i)} features")
    
    # Main simulation loop
    for k in range(n_steps - 1):
        # Generate true next state
        x_true_ekf[k+1] = plant(x_true_ekf[k], u_hist[k], dt)
        
        # Create noisy measurement
        measurement_noise = np.array([
            np.random.laplace(0, position_noise_std/np.sqrt(2)),
            np.random.laplace(0, position_noise_std/np.sqrt(2)),
            np.random.standard_cauchy() * angle_noise_std * 0.5,
            np.random.exponential(velocity_noise_std) - velocity_noise_std,
            np.random.uniform(-omega_noise_std*np.sqrt(3), omega_noise_std*np.sqrt(3))
        ])
        y_measured_ekf[k+1] = x_true_ekf[k+1] + measurement_noise
        
        # EKF Update & Predict
        start_time = time.perf_counter()
        ekf.update(y_measured_ekf[k+1], x_est_ekf[k], u_hist[k])
        ekf_time = time.perf_counter() - start_time
        ekf_training_times.append(ekf_time)
        
        # Predict next state
        for i in range(5):
            z_ekf = construct_z_vector(x_est_ekf[k], u_hist[k], i)
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z_ekf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            err_ekf = np.linalg.norm(x_true_ekf[k] - x_est_ekf[k])
            print(f"[EKF] Step {k}/{n_steps-1} - Error: {err_ekf:.6f}")
    
    # Calculate statistics
    print("\n" + "="*70)
    print("📊 [EKF] RESULTS")
    print("="*70)
    
    mse_x_ekf = np.mean((x_true_ekf[:, 0] - x_est_ekf[:, 0])**2)
    mse_y_ekf = np.mean((x_true_ekf[:, 1] - x_est_ekf[:, 1])**2)
    mse_θ_ekf = np.mean((x_true_ekf[:, 2] - x_est_ekf[:, 2])**2)
    mse_v_ekf = np.mean((x_true_ekf[:, 3] - x_est_ekf[:, 3])**2)
    mse_ω_ekf = np.mean((x_true_ekf[:, 4] - x_est_ekf[:, 4])**2)
    mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_θ_ekf + mse_v_ekf + mse_ω_ekf
    
    print(f"\nMSE total: {mse_total_ekf:.8f}")
    print(f"  x: {mse_x_ekf:.8f} | y: {mse_y_ekf:.8f} | θ: {mse_θ_ekf:.8f}")
    print(f"  v: {mse_v_ekf:.8f} | ω: {mse_ω_ekf:.8f}")
    
    ekf_total_time = np.sum(ekf_training_times)
    ekf_mean_time = np.mean(ekf_training_times)
    ekf_std_time = np.std(ekf_training_times)
    
    print(f"\n⏱️  Training time statistics:")
    print(f"  Total: {ekf_total_time:.6f} s")
    print(f"  Mean: {ekf_mean_time*1000:.3f} ms")
    print(f"  Std: {ekf_std_time*1000:.3f} ms")
    print(f"  Min: {np.min(ekf_training_times)*1000:.3f} ms")
    print(f"  Max: {np.max(ekf_training_times)*1000:.3f} ms")
    print("="*70)


[EKF] Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(4,), min=-0.6079, max=-0.1390, mean=-0.4011
  Neuron 1: shape=(3,), min=-0.9538, max=-0.2944, mean=-0.6189
  Neuron 2: shape=(4,), min=-0.5535, max=0.7071, mean=0.1354
  Neuron 3: shape=(5,), min=-0.9177, max=0.8417, mean=-0.2119
  Neuron 4: shape=(4,), min=-0.9594, max=0.9647, mean=-0.2000

✓ EKF initialized with uniform RHONN weights

[EKF] Starting simulation...

Neuron structure:
  Neuron 0 (x (pos)): 4 features
  Neuron 1 (y (pos)): 3 features
  Neuron 2 (θ (orient)): 4 features
  Neuron 3 (v (lin vel)): 5 features
  Neuron 4 (ω (ang vel)): 4 features
[EKF] Step 300/999 - Error: 0.029412
[EKF] Step 600/999 - Error: 0.025024
[EKF] Step 900/999 - Error: 0.024423

📊 [EKF] RESULTS

MSE total: 0.00127124
  x: 0.00022338 | y: 0.00026484 | θ: 0.00025778
  v: 0.00015047 | ω: 0.00037477

⏱️  Training time statistics:
  Total: 0.141737 s
  Mean: 0.142 ms
  Std: 0.035 ms
  Min: 0.110 ms
  Max: 0.580 ms


In [4]:
print("\n\n" + "="*80)
print(" "*20 + "SIMULATION 2: UKF-RHONN ONLY")
print("="*80)



                    SIMULATION 2: UKF-RHONN ONLY


In [5]:
# ============================================================
# SIMULATION 2: UKF-RHONN ONLY
# ============================================================
if __name__ == "__main__":
    # Simulation parameters (same as EKF)
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    n_states = 5
    
    # Measurement noise parameters (same as EKF)
    position_noise_std = 0.01
    angle_noise_std = 0.005
    velocity_noise_std = 0.02
    omega_noise_std = 0.003
    
    # Set independent random seed for UKF simulation
    np.random.seed(2002)  # Unique seed for UKF
    
    # Generate initial weights (uniform distribution)
    initial_weights_ukf = []
    for i in range(n_states):
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights_ukf.append(w_init.copy())
    
    print("\n[UKF] Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights_ukf):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}, mean={w.mean():.4f}")
    
    # Initialize UKF trainer
    ukf = UKF_Trainer(n_states, eta=1.0, alpha=1e-2)
    
    # Set initial weights
    for i in range(n_states):
        ukf.weights[i] = initial_weights_ukf[i].copy()
    
    print("\n✓ UKF initialized with uniform RHONN weights")
    
    # Arrays for states
    x_true_ukf = np.zeros((n_steps, 5))
    x_est_ukf = np.zeros((n_steps, 5))
    y_measured_ukf = np.zeros((n_steps, 5))
    ukf_training_times = []
    
    # Initial conditions
    x_true_ukf[0] = [0.0, 0.0, 0.0, 0.0, 0.0]
    x_est_ukf[0] = x_true_ukf[0]
    
    # Add noise to initial measurement
    initial_noise = np.array([
        np.random.normal(0, position_noise_std),
        np.random.normal(0, position_noise_std),
        np.random.normal(0, angle_noise_std),
        np.random.normal(0, velocity_noise_std),
        np.random.normal(0, omega_noise_std)
    ])
    y_measured_ukf[0] = x_true_ukf[0] + initial_noise
    
    # Excitation input (same as EKF)
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        v_cmd = 1.0 + 0.5 * np.sin(0.5 * t[k])
        ω_cmd = 0.8 * np.sin(1.0 * t[k])
        
        if 300 < k < 320:
            v_cmd = 2.0
            ω_cmd = 1.5
        elif 700 < k < 720:
            v_cmd = 0.5
            ω_cmd = -1.2
        elif 1100 < k < 1120:
            v_cmd = -0.8
            ω_cmd = 0.0
        
        u_hist[k] = [v_cmd, ω_cmd]
    
    print("\n[UKF] Starting simulation...")
    state_names = ['x (pos)', 'y (pos)', 'θ (orient)', 'v (lin vel)', 'ω (ang vel)']
    print("\nNeuron structure:")
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}): {get_z_size(i)} features")
    
    # Main simulation loop
    for k in range(n_steps - 1):
        # Generate true next state
        x_true_ukf[k+1] = plant(x_true_ukf[k], u_hist[k], dt)
        
        # Create noisy measurement
        measurement_noise = np.array([
            np.random.laplace(0, position_noise_std/np.sqrt(2)),
            np.random.laplace(0, position_noise_std/np.sqrt(2)),
            np.random.standard_cauchy() * angle_noise_std * 0.5,
            np.random.exponential(velocity_noise_std) - velocity_noise_std,
            np.random.uniform(-omega_noise_std*np.sqrt(3), omega_noise_std*np.sqrt(3))
        ])
        y_measured_ukf[k+1] = x_true_ukf[k+1] + measurement_noise
        
        # UKF Update & Predict
        start_time = time.perf_counter()
        ukf.update(y_measured_ukf[k+1], x_est_ukf[k], u_hist[k])
        ukf_time = time.perf_counter() - start_time
        ukf_training_times.append(ukf_time)
        
        # Predict next state
        for i in range(5):
            z_ukf = construct_z_vector(x_est_ukf[k], u_hist[k], i)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z_ukf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            err_ukf = np.linalg.norm(x_true_ukf[k] - x_est_ukf[k])
            print(f"[UKF] Step {k}/{n_steps-1} - Error: {err_ukf:.6f}")
    
    # Calculate statistics
    print("\n" + "="*70)
    print("📊 [UKF] RESULTS")
    print("="*70)
    
    mse_x_ukf = np.mean((x_true_ukf[:, 0] - x_est_ukf[:, 0])**2)
    mse_y_ukf = np.mean((x_true_ukf[:, 1] - x_est_ukf[:, 1])**2)
    mse_θ_ukf = np.mean((x_true_ukf[:, 2] - x_est_ukf[:, 2])**2)
    mse_v_ukf = np.mean((x_true_ukf[:, 3] - x_est_ukf[:, 3])**2)
    mse_ω_ukf = np.mean((x_true_ukf[:, 4] - x_est_ukf[:, 4])**2)
    mse_total_ukf = mse_x_ukf + mse_y_ukf + mse_θ_ukf + mse_v_ukf + mse_ω_ukf
    
    print(f"\nMSE total: {mse_total_ukf:.8f}")
    print(f"  x: {mse_x_ukf:.8f} | y: {mse_y_ukf:.8f} | θ: {mse_θ_ukf:.8f}")
    print(f"  v: {mse_v_ukf:.8f} | ω: {mse_ω_ukf:.8f}")
    
    ukf_total_time = np.sum(ukf_training_times)
    ukf_mean_time = np.mean(ukf_training_times)
    ukf_std_time = np.std(ukf_training_times)
    
    print(f"\n⏱️  Training time statistics:")
    print(f"  Total: {ukf_total_time:.6f} s")
    print(f"  Mean: {ukf_mean_time*1000:.3f} ms")
    print(f"  Std: {ukf_std_time*1000:.3f} ms")
    print(f"  Min: {np.min(ukf_training_times)*1000:.3f} ms")
    print(f"  Max: {np.max(ukf_training_times)*1000:.3f} ms")
    print("="*70)


[UKF] Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(4,), min=-0.4054, max=0.3090, mean=-0.0049
  Neuron 1: shape=(3,), min=-0.7967, max=0.4893, mean=0.0136
  Neuron 2: shape=(4,), min=-0.9058, max=-0.2150, mean=-0.7001
  Neuron 3: shape=(5,), min=-0.6400, max=0.4365, mean=-0.1636
  Neuron 4: shape=(4,), min=-0.3571, max=0.9631, mean=0.4000

✓ UKF initialized with uniform RHONN weights

[UKF] Starting simulation...

Neuron structure:
  Neuron 0 (x (pos)): 4 features
  Neuron 1 (y (pos)): 3 features
  Neuron 2 (θ (orient)): 4 features
  Neuron 3 (v (lin vel)): 5 features
  Neuron 4 (ω (ang vel)): 4 features
[UKF] Step 300/999 - Error: 0.014224
[UKF] Step 600/999 - Error: 0.024243
[UKF] Step 900/999 - Error: 0.009091

📊 [UKF] RESULTS

MSE total: 0.00078135
  x: 0.00006284 | y: 0.00009504 | θ: 0.00034899
  v: 0.00015629 | ω: 0.00011819

⏱️  Training time statistics:
  Total: 0.235422 s
  Mean: 0.236 ms
  Std: 0.033 ms
  Min: 0.213 ms
  Max: 1.007 ms


In [6]:
print("\n\n" + "="*80)
print(" "*20 + "SIMULATION 3: PF-RHONN ONLY")
print("="*80)



                    SIMULATION 3: PF-RHONN ONLY


In [7]:
# ============================================================
# SIMULATION 3: PF-RHONN ONLY
# ============================================================
if __name__ == "__main__":
    # Simulation parameters (same as EKF and UKF)
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    n_states = 5
    
    # Measurement noise parameters (same as others)
    position_noise_std = 0.01
    angle_noise_std = 0.005
    velocity_noise_std = 0.02
    omega_noise_std = 0.003
    
    # Set independent random seed for PF simulation
    np.random.seed(3003)  # Unique seed for PF
    
    # Generate initial weights (uniform distribution)
    initial_weights_pf = []
    for i in range(n_states):
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights_pf.append(w_init.copy())
    
    print("\n[PF] Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights_pf):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}, mean={w.mean():.4f}")
    
    # Initialize PF trainer
    pf = PF_Trainer(n_states, n_particles=600)
    
    # Set initial weights (all particles start with same weights)
    for i in range(n_states):
        pf.particles[i] = np.tile(initial_weights_pf[i], (pf.n_particles, 1))
    
    print("\n✓ PF initialized with uniform RHONN weights")
    
    # Arrays for states
    x_true_pf = np.zeros((n_steps, 5))
    x_est_pf = np.zeros((n_steps, 5))
    y_measured_pf = np.zeros((n_steps, 5))
    pf_training_times = []
    
    # Initial conditions
    x_true_pf[0] = [0.0, 0.0, 0.0, 0.0, 0.0]
    x_est_pf[0] = x_true_pf[0]
    
    # Add noise to initial measurement
    initial_noise = np.array([
        np.random.normal(0, position_noise_std),
        np.random.normal(0, position_noise_std),
        np.random.normal(0, angle_noise_std),
        np.random.normal(0, velocity_noise_std),
        np.random.normal(0, omega_noise_std)
    ])
    y_measured_pf[0] = x_true_pf[0] + initial_noise
    
    # Excitation input (same as others)
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        v_cmd = 1.0 + 0.5 * np.sin(0.5 * t[k])
        ω_cmd = 0.8 * np.sin(1.0 * t[k])
        
        if 300 < k < 320:
            v_cmd = 2.0
            ω_cmd = 1.5
        elif 700 < k < 720:
            v_cmd = 0.5
            ω_cmd = -1.2
        elif 1100 < k < 1120:
            v_cmd = -0.8
            ω_cmd = 0.0
        
        u_hist[k] = [v_cmd, ω_cmd]
    
    print("\n[PF] Starting simulation...")
    state_names = ['x (pos)', 'y (pos)', 'θ (orient)', 'v (lin vel)', 'ω (ang vel)']
    print("\nNeuron structure:")
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}): {get_z_size(i)} features")
    
    # Main simulation loop
    for k in range(n_steps - 1):
        # Generate true next state
        x_true_pf[k+1] = plant(x_true_pf[k], u_hist[k], dt)
        
        # Create noisy measurement
        measurement_noise = np.array([
            np.random.laplace(0, position_noise_std/np.sqrt(2)),
            np.random.laplace(0, position_noise_std/np.sqrt(2)),
            np.random.standard_cauchy() * angle_noise_std * 0.5,
            np.random.exponential(velocity_noise_std) - velocity_noise_std,
            np.random.uniform(-omega_noise_std*np.sqrt(3), omega_noise_std*np.sqrt(3))
        ])
        y_measured_pf[k+1] = x_true_pf[k+1] + measurement_noise
        
        # PF Update & Predict
        start_time = time.perf_counter()
        pf.update(y_measured_pf[k+1], x_est_pf[k], u_hist[k])
        pf_time = time.perf_counter() - start_time
        pf_training_times.append(pf_time)
        
        w_pf = pf.get_estimates()
        # Predict next state
        for i in range(5):
            z_pf = construct_z_vector(x_est_pf[k], u_hist[k], i)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z_pf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            err_pf = np.linalg.norm(x_true_pf[k] - x_est_pf[k])
            print(f"[PF] Step {k}/{n_steps-1} - Error: {err_pf:.6f}")
    
    # Calculate statistics
    print("\n" + "="*70)
    print("📊 [PF] RESULTS")
    print("="*70)
    
    mse_x_pf = np.mean((x_true_pf[:, 0] - x_est_pf[:, 0])**2)
    mse_y_pf = np.mean((x_true_pf[:, 1] - x_est_pf[:, 1])**2)
    mse_θ_pf = np.mean((x_true_pf[:, 2] - x_est_pf[:, 2])**2)
    mse_v_pf = np.mean((x_true_pf[:, 3] - x_est_pf[:, 3])**2)
    mse_ω_pf = np.mean((x_true_pf[:, 4] - x_est_pf[:, 4])**2)
    mse_total_pf = mse_x_pf + mse_y_pf + mse_θ_pf + mse_v_pf + mse_ω_pf
    
    print(f"\nMSE total: {mse_total_pf:.8f}")
    print(f"  x: {mse_x_pf:.8f} | y: {mse_y_pf:.8f} | θ: {mse_θ_pf:.8f}")
    print(f"  v: {mse_v_pf:.8f} | ω: {mse_ω_pf:.8f}")
    
    pf_total_time = np.sum(pf_training_times)
    pf_mean_time = np.mean(pf_training_times)
    pf_std_time = np.std(pf_training_times)
    
    print(f"\n⏱️  Training time statistics:")
    print(f"  Total: {pf_total_time:.6f} s")
    print(f"  Mean: {pf_mean_time*1000:.3f} ms")
    print(f"  Std: {pf_std_time*1000:.3f} ms")
    print(f"  Min: {np.min(pf_training_times)*1000:.3f} ms")
    print(f"  Max: {np.max(pf_training_times)*1000:.3f} ms")
    print("="*70)


[PF] Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(4,), min=-0.7758, max=0.4107, mean=-0.3347
  Neuron 1: shape=(3,), min=-0.2076, max=0.4940, mean=0.1217
  Neuron 2: shape=(4,), min=-0.6730, max=-0.1726, mean=-0.4258
  Neuron 3: shape=(5,), min=-0.4860, max=0.5011, mean=0.1269
  Neuron 4: shape=(4,), min=-0.5606, max=0.6923, mean=0.1759

✓ PF initialized with uniform RHONN weights

[PF] Starting simulation...

Neuron structure:
  Neuron 0 (x (pos)): 4 features
  Neuron 1 (y (pos)): 3 features
  Neuron 2 (θ (orient)): 4 features
  Neuron 3 (v (lin vel)): 5 features
  Neuron 4 (ω (ang vel)): 4 features
[PF] Step 300/999 - Error: 0.040474
[PF] Step 600/999 - Error: 0.022304
[PF] Step 900/999 - Error: 0.019329

📊 [PF] RESULTS

MSE total: 0.00257483
  x: 0.00027192 | y: 0.00008063 | θ: 0.00185839
  v: 0.00021991 | ω: 0.00014398

⏱️  Training time statistics:
  Total: 0.388254 s
  Mean: 0.389 ms
  Std: 0.060 ms
  Min: 0.305 ms
  Max: 0.601 ms
